C++ derleyicisinin kurulu olup olmadığını kontrol eder.

In [1]:
!g++ --version

g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



OpenCV'nin kurulu olup olmadığını ve sürümünü kontrol eder.

In [2]:
!pkg-config --modversion opencv4

Package opencv4 was not found in the pkg-config search path.
Perhaps you should add the directory containing `opencv4.pc'
to the PKG_CONFIG_PATH environment variable
Package 'opencv4', required by 'virtual:world', not found


Bu işlem OpenCV'nin C++ geliştirme kütüphanelerini yükleyecek. Birkaç dakika sürebilir.

In [3]:
!apt-get update -qq && apt-get install -y -qq libopencv-dev

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package libevdev2:amd64.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../000-libevdev2_1.13.1+dfsg-1build1_amd64.deb ...
Unpacking libevdev2:amd64 (1.13.1+dfsg-1build1) ...
Selecting previously unselected package at-spi2-common.
Preparing to unpack .../001-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../002-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatspi2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../003-libxtst6_2%3a1.2.3-1.1build1_amd64.deb ...
Unpacking libxtst6:amd64 

In [4]:
!pkg-config --modversion opencv4

4.6.0


In [5]:
!ls -l /content/image1.png

-rw-r--r-- 1 root root 151102 Oct 11 09:54 /content/image1.png


C++ koduyla görüntüyü oku

In [18]:
%%writefile hafta2_1.cpp
#include <opencv2/opencv.hpp>
#include <iostream>

int main() {
    cv::Mat image = cv::imread("/content/image1.png");

    if (image.empty()) {
        std::cout << "Goruntu okunamadi!" << std::endl;
        return 1;
    }

    std::cout << "Goruntu basariyla okundu!" << std::endl;
    std::cout << "Genislik: " << image.cols << std::endl;
    std::cout << "Yukseklik: " << image.rows << std::endl;

    return 0;
}

Overwriting hafta2_1.cpp


derleyip çalıştırırma

In [19]:
!g++ hafta2_1.cpp -o hafta2_1 $(pkg-config --cflags --libs opencv4)

In [20]:
!./hafta2_1

Goruntu basariyla okundu!
Genislik: 325
Yukseklik: 350


Görüntüyü 1024×768 boyutuna yeniden boyutlandır

In [21]:
%%writefile hafta2_1.cpp
#include <opencv2/opencv.hpp>
#include <iostream>

int main() {
    cv::Mat image = cv::imread("/content/image1.png");

    if (image.empty()) {
        std::cout << "Goruntu okunamadi!" << std::endl;
        return 1;
    }

    cv::Mat resizedImage;

    cv::resize(image, resizedImage, cv::Size(1024, 768));

    if (cv::imwrite("/content/resized_image.png", resizedImage)) {
    std::cout << "Goruntu basariyla kaydedildi!" << std::endl;
    } else {
        std::cout << "Goruntu kaydedilemedi!" << std::endl;
    }

    std::cout << "Yeni genislik: " << resizedImage.cols << std::endl;
    std::cout << "Yeni yukseklik: " << resizedImage.rows << std::endl;

    return 0;
}

Overwriting hafta2_1.cpp


In [22]:
!g++ hafta2_1.cpp -o hafta2_1 $(pkg-config --cflags --libs opencv4) && ./hafta2_1

Goruntu basariyla kaydedildi!
Yeni genislik: 1024
Yeni yukseklik: 768


**Hafta2_2 : Tek kanallı (gri seviye) görüntü üzerinde 8-bit'ten 6-bit'e parlaklık nicemleme (quantization) işlemi (paylaşılan görsel kullanılabilir).**

In [23]:
%%writefile hafta2_2.cpp
#include <opencv2/opencv.hpp>
#include <iostream>

int main() {
    cv::Mat image = cv::imread("/content/image1.png", cv::IMREAD_GRAYSCALE );

    if (image.empty()) {
        std::cout << "Image could not be read!" << std::endl;
        return 1;
    }

    cv::imwrite("/content/hafta2_2_original_8bit.png", image);

    cv::Mat quantized6bit = image / 4;

    cv::Mat display6bit = quantized6bit * 4;

    cv::imwrite("/content/hafta2_2_quantized_6bit.png", display6bit);

    std::cout << "Both images saved successfully!" << std::endl;

    return 0;
}

Overwriting hafta2_2.cpp


In [24]:
!g++ hafta2_2.cpp -o hafta2_2 $(pkg-config --cflags --libs opencv4) && ./hafta2_2

Both images saved successfully!


**Hafta2_3 : Görüntünün 4 parçaya bölünerek 4 ayrı iş parçacığı (thread) ile farklı parlaklık değerlerine ayarlanması.**

In [25]:
%%writefile hafta2_3.cpp
#include <opencv2/opencv.hpp>
#include <iostream>
#include <thread>
using namespace cv;
using namespace std;

void parlaklık_Ayarlama_Bolgede(Mat& img, int x1, int y1, int x2, int y2, int value) {
    for (int y = y1; y < y2; y++) {
        for (int x = x1; x < x2; x++) {
            int pixel = img.at<uchar>(y, x) + value;

            if (pixel > 255) pixel = 255;
            if (pixel < 0) pixel = 0;

            img.at<uchar>(y, x) = pixel;
        }
    }
}

int main() {
    Mat img = imread("/content/image1.png", IMREAD_GRAYSCALE);

    if (img.empty()) {
        cout << "Image not found!" << endl;
        return -1;
    }

    int w = img.cols;
    int h = img.rows;
    int midX = w / 2;
    int midY = h / 2;

    thread t1(parlaklık_Ayarlama_Bolgede, ref(img), 0, 0, midX, midY, -60);
    thread t2(parlaklık_Ayarlama_Bolgede, ref(img), midX, 0, w, midY, -20);
    thread t3(parlaklık_Ayarlama_Bolgede, ref(img), 0, midY, midX, h, 20);
    thread t4(parlaklık_Ayarlama_Bolgede, ref(img), midX, midY, w, h, 60);

    t1.join();
    t2.join();
    t3.join();
    t4.join();

    imwrite("/content/hafta2_3_threads.png", img);
    cout << "Result saved!" << endl;

    return 0;
}

Writing hafta2_3.cpp


In [26]:
!g++ hafta2_3.cpp -o hafta2_3 $(pkg-config --cflags --libs opencv4) -pthread
!./hafta2_3

Result saved!


**Hafta_2_4 : Parlaklık değiştirme işleminin tek iş parçacığı (1 thread) ile çoklu iş parçacığı (4 thread) arasındaki çalışma süresi farkının karşılaştırılması.**

In [32]:
%%writefile hafta_2_4.cpp
#include <opencv2/opencv.hpp>
#include <iostream>
#include <thread>
#include <chrono>
using namespace cv;
using namespace std;
using namespace chrono;

void parlaklık_Ayarlama(Mat& img, int x1, int y1, int x2, int y2, int value) {
    for (int y = y1; y < y2; y++) {
        for (int x = x1; x < x2; x++) {
            int pixel = img.at<uchar>(y, x) + value;
            if (pixel > 255) pixel = 255;
            if (pixel < 0) pixel = 0;
            img.at<uchar>(y, x) = pixel;
        }
    }
}

int main() {
    Mat original = imread("/content/image1.png", IMREAD_GRAYSCALE);
    if (original.empty()) {
        cout << "Image not found!" << endl;
        return -1;
    }

    int w = original.cols, h = original.rows;
    int mx = w / 2, my = h / 2;

    // 1 thread
    Mat img1 = original.clone();
    auto start1 = high_resolution_clock::now();

    parlaklık_Ayarlama(img1, 0, 0, mx, my, -60);
    parlaklık_Ayarlama(img1, mx, 0, w, my, -20);
    parlaklık_Ayarlama(img1, 0, my, mx, h, 20);
    parlaklık_Ayarlama(img1, mx, my, w, h, 60);

    auto end1 = high_resolution_clock::now();

    // 4 threads
    Mat img4 = original.clone();
    auto start4 = high_resolution_clock::now();

    thread t1(parlaklık_Ayarlama, ref(img4), 0, 0, mx, my, -60);
    thread t2(parlaklık_Ayarlama, ref(img4), mx, 0, w, my, -20);
    thread t3(parlaklık_Ayarlama, ref(img4), 0, my, mx, h, 20);
    thread t4(parlaklık_Ayarlama, ref(img4), mx, my, w, h, 60);

    t1.join();
    t2.join();
    t3.join();
    t4.join();

    auto end4 = high_resolution_clock::now();

    double time1 = duration<double, milli>(end1 - start1).count();
    double time4 = duration<double, milli>(end4 - start4).count();

    cout << "1 thread: " << time1 << " ms" << endl;
    cout << "4 threads: " << time4 << " ms" << endl;

    imwrite("/content/hafta_2_4_1thread.png", img1);
    imwrite("/content/hafta_2_4_4threads.png", img4);

    return 0;
}

Overwriting hafta_2_4.cpp


In [33]:
!g++ hafta_2_4.cpp -o hafta_2_4 $(pkg-config --cflags --libs opencv4) -pthread
!./hafta_2_4

1 thread: 4.44513 ms
4 threads: 3.23087 ms


**Hafta_2_5 : Görüntüyü 4 parçaya bölerek her parçanın histogramının 4 thred ile paralel hesaplanması ve birleştirilerek ana histogramın elde edilmesi.**

In [36]:
%%writefile hafta_2_5.cpp
#include <opencv2/opencv.hpp>
#include <iostream>
#include <thread>
using namespace cv;
using namespace std;

void calculateHistogram(Mat& img, int x1, int y1, int x2, int y2, int hist[256]) {
    for (int i = 0; i < 256; i++)
        hist[i] = 0;

    for (int y = y1; y < y2; y++) {
        for (int x = x1; x < x2; x++) {
            int pixel = img.at<uchar>(y, x);
            hist[pixel]++;
        }
    }
}

int main() {
    Mat img = imread("/content/image1.png", IMREAD_GRAYSCALE);
    if (img.empty()) {
        cout << "Image not found!" << endl;
        return -1;
    }

    int w = img.cols, h = img.rows;
    int mx = w / 2, my = h / 2;

    int hist1[256], hist2[256], hist3[256], hist4[256];
    int mainHist[256] = {0};

    thread t1(calculateHistogram, ref(img), 0, 0, mx, my, hist1);
    thread t2(calculateHistogram, ref(img), mx, 0, w, my, hist2);
    thread t3(calculateHistogram, ref(img), 0, my, mx, h, hist3);
    thread t4(calculateHistogram, ref(img), mx, my, w, h, hist4);

    t1.join();
    t2.join();
    t3.join();
    t4.join();

    for (int i = 0; i < 256; i++)
        mainHist[i] = hist1[i] + hist2[i] + hist3[i] + hist4[i];

    cout << "Intensity\tPixel Count" << endl;

    for (int i = 0; i < 256; i++) {
        cout << i << "\t\t" << mainHist[i] << endl;
    }

    return 0;
}

Overwriting hafta_2_5.cpp


In [37]:
!g++ hafta_2_5.cpp -o hafta_2_5 $(pkg-config --cflags --libs opencv4) -pthread
!./hafta_2_5

Intensity	Pixel Count
0		312
1		0
2		0
3		0
4		0
5		0
6		0
7		0
8		0
9		0
10		0
11		0
12		0
13		0
14		0
15		0
16		0
17		0
18		0
19		0
20		0
21		595
22		0
23		0
24		0
25		0
26		0
27		0
28		400
29		0
30		0
31		0
32		0
33		0
34		269
35		0
36		0
37		0
38		0
39		421
40		0
41		0
42		0
43		324
44		0
45		0
46		523
47		0
48		0
49		0
50		568
51		0
52		0
53		679
54		0
55		0
56		1001
57		0
58		0
59		1259
60		0
61		1649
62		0
63		0
64		1233
65		0
66		1660
67		0
68		3316
69		0
70		3627
71		0
72		3951
73		0
74		3195
75		0
76		2634
77		0
78		2427
79		0
80		2809
81		0
82		2638
83		0
84		2273
85		2797
86		0
87		2530
88		0
89		3256
90		3058
91		0
92		2781
93		2508
94		0
95		1781
96		1969
97		0
98		1973
99		1414
100		0
101		1565
102		1418
103		1398
104		0
105		1382
106		1286
107		1331
108		0
109		1443
110		1434
111		1221
112		1207
113		0
114		1401
115		1123
116		1013
117		716
118		707
119		801
120		535
121		0
122		628
123		410
124		358
125		346
126		343
127		316
128		306
129		298
130		289
131		266
132		23

**Hafta_2_6 : Histogram hesaplama işleminin tek thread ve 4 thread arasındaki süre farkının karşılaştırılması.**

In [40]:

%%writefile hafta_2_6.cpp
#include <opencv2/opencv.hpp>
#include <iostream>
#include <thread>
#include <chrono>
using namespace cv;
using namespace std;
using namespace chrono;

void calculateHistogram(Mat& img, int hist[256]) {
    for (int i = 0; i < 256; i++)
        hist[i] = 0;

    for (int y = 0; y < img.rows; y++) {
        for (int x = 0; x < img.cols; x++) {
            hist[img.at<uchar>(y, x)]++;
        }
    }
}

int main() {
    Mat img = imread("/content/image1.png", IMREAD_GRAYSCALE);

    if (img.empty()) {
        cout << "Image not found!" << endl;
        return -1;
    }

    int w = img.cols, h = img.rows;
    int mx = w / 2, my = h / 2;

    Mat sub_img1 = img(Rect(0, 0, mx, my));
    Mat sub_img2 = img(Rect(mx, 0, w - mx, my));
    Mat sub_img3 = img(Rect(0, my, mx, h - my));
    Mat sub_img4 = img(Rect(mx, my, w - mx, h - my));

    int hist1[256], hist2[256], hist3[256], hist4[256];
    int mainHist[256] = {0};

    int hist_1_thread[256];

    auto start1 = high_resolution_clock::now();
    calculateHistogram(img, hist_1_thread);
    auto end1 = high_resolution_clock::now();

    auto start4 = high_resolution_clock::now();

    thread t1(calculateHistogram, ref(sub_img1), hist1);
    thread t2(calculateHistogram, ref(sub_img2), hist2);
    thread t3(calculateHistogram, ref(sub_img3), hist3);
    thread t4(calculateHistogram, ref(sub_img4), hist4);

    t1.join();
    t2.join();
    t3.join();
    t4.join();

    for (int i = 0; i < 256; i++) {
        mainHist[i] = hist1[i] + hist2[i] + hist3[i] + hist4[i];
    }

    auto end4 = high_resolution_clock::now();

    double time1 = duration<double, milli>(end1 - start1).count();
    double time4 = duration<double, milli>(end4 - start4).count();

    cout << "1 thread: " << time1 << " ms" << endl;
    cout << "4 threads: " << time4 << " ms" << endl;

    cout << "\nIntensity \t1 Thread Hist \t4 Threads Hist  \n";
    for (int i = 0; i < 256; i++) {
        cout << i << "\t\t" << mainHist[i] << "\t\t" << hist_1_thread[i] << endl;
    }

    return 0;
}


Overwriting hafta_2_6.cpp


In [41]:
!g++ hafta_2_6.cpp -o hafta_2_6 $(pkg-config --cflags --libs opencv4) -pthread
!./hafta_2_6

1 thread: 1.00351 ms
4 threads: 1.50762 ms

Intensity 	1 Thread Hist 	4 Threads Hist  
0		312		312
1		0		0
2		0		0
3		0		0
4		0		0
5		0		0
6		0		0
7		0		0
8		0		0
9		0		0
10		0		0
11		0		0
12		0		0
13		0		0
14		0		0
15		0		0
16		0		0
17		0		0
18		0		0
19		0		0
20		0		0
21		595		595
22		0		0
23		0		0
24		0		0
25		0		0
26		0		0
27		0		0
28		400		400
29		0		0
30		0		0
31		0		0
32		0		0
33		0		0
34		269		269
35		0		0
36		0		0
37		0		0
38		0		0
39		421		421
40		0		0
41		0		0
42		0		0
43		324		324
44		0		0
45		0		0
46		523		523
47		0		0
48		0		0
49		0		0
50		568		568
51		0		0
52		0		0
53		679		679
54		0		0
55		0		0
56		1001		1001
57		0		0
58		0		0
59		1259		1259
60		0		0
61		1649		1649
62		0		0
63		0		0
64		1233		1233
65		0		0
66		1660		1660
67		0		0
68		3316		3316
69		0		0
70		3627		3627
71		0		0
72		3951		3951
73		0		0
74		3195		3195
75		0		0
76		2634		2634
77		0		0
78		2427		2427
79		0		0
80		2809		2809
81		0		0
82		2638		2638
83		0		0
84		2273		2273
85		2797		2797
86		0		0
87		2530		2530

**Hafta_2_7 : Görüntü piksel değerlerini 0.75 ile çarpıp 20 ekleme işlemi; elde edilen sonucun sadece 0.75 ile çarpılan durumla görsel/sayısal olarak karşılaştırılması.**

In [43]:

%%writefile hafta_2_7.cpp
#include <opencv2/opencv.hpp>
#include <iostream>
using namespace cv;
using namespace std;

void multiplyImage(Mat& img, double value) {
    for (int y = 0; y < img.rows; y++) {
        for (int x = 0; x < img.cols; x++) {
            int pixel = img.at<uchar>(y, x);
            int result = pixel * value;

            if (result > 255) result = 255;
            if (result < 0) result = 0;

            img.at<uchar>(y, x) = result;
        }
    }
}

void addImage(Mat& img, int value) {
    for (int y = 0; y < img.rows; y++) {
        for (int x = 0; x < img.cols; x++) {
            int pixel = img.at<uchar>(y, x);
            int result = pixel + value;

            if (result > 255) result = 255;
            if (result < 0) result = 0;

            img.at<uchar>(y, x) = result;
        }
    }
}

int main() {
    Mat img = imread("/content/image1.png", IMREAD_GRAYSCALE);

    if (img.empty()) {
        cout << "Image not found!" << endl;
        return -1;
    }

    // First image: only multiply by 0.75
    Mat img1 = img.clone();
    multiplyImage(img1, 0.75);

    // Second image: multiply by 0.75, then add 20
    Mat img2 = img.clone();
    multiplyImage(img2, 0.75);
    addImage(img2, 20);

    imwrite("/content/hafta_2_7_multiply.png", img1);
    imwrite("/content/hafta_2_7_multiply_add.png", img2);

    cout << "Original average: " << mean(img)[0] << endl;
    cout << "Multiply average: " << mean(img1)[0] << endl;
    cout << "Multiply + add average: " << mean(img2)[0] << endl;

    return 0;
}


Overwriting hafta_2_7.cpp


In [44]:
!g++ hafta_2_7.cpp -o hafta_2_7 $(pkg-config --cflags --libs opencv4)
!./hafta_2_7

Original average: 112.55
Multiply average: 84.056
Multiply + add average: 104.056


## **HAFTA 3**

In [46]:

%%writefile hafta_3_1.cpp
#include <opencv2/opencv.hpp>
#include <iostream>
using namespace cv;
using namespace std;

int main() {
    Mat img = imread("/content/hafta_3.png", IMREAD_GRAYSCALE);

    if (img.empty()) {
        cout << "Error: Image not found!" << endl;
        return 1;
    }

    int minVal = 255;
    int maxVal = 0;

    for (int y = 0; y < img.rows; y++) {
        for (int x = 0; x < img.cols; x++) {
            int pixel = img.at<uchar>(y, x);

            if (pixel < minVal)
                minVal = pixel;

            if (pixel > maxVal)
                maxVal = pixel;
        }
    }

    cout << "Minimum value: " << minVal << endl;
    cout << "Maximum value: " << maxVal << endl;

    Mat result(img.rows, img.cols, CV_8UC1);

    if (maxVal == minVal) {
        cout << "All pixels have the same value; cannot scale." << endl;
        return 1;
    }

    for (int y = 0; y < img.rows; y++) {
        for (int x = 0; x < img.cols; x++) {
            int oldPixel = img.at<uchar>(y, x);

            double newPixel =
                (double)(oldPixel - minVal) * 255.0
                / (maxVal - minVal);

            result.at<uchar>(y, x) = (uchar)(newPixel + 0.5);
        }
    }

    imwrite("/content/hafta_3_1_contrast_stretched.png", result);

    return 0;
}


Overwriting hafta_3_1.cpp


In [47]:
!g++ hafta_3_1.cpp -o hafta_3_1 `pkg-config --cflags --libs opencv4`
!./hafta_3_1

Minimum value: 100
Maximum value: 255


**Hafta_3_2 : Düşük kontrastlı ve tek kanallı bir görüntüyü OpenCV kullanarak açınız. Bu görüntüdeki değerlerin 256-değerli bir histogramını çıkarınız. Histogramdaki değerleri kullanarak Kümülatif Dağılım Değerlerini hesaplayınız. Bu dağılımı kullanarah histogram eşitleme (histogram equization) işlemini gerçekleştiriniz. Çıktı resmi kaydediniz. Histogram çıkartma, CDF ve eşitleme işlemlerinde hazır fonksiyon kullanmayınız.**

In [49]:

%%writefile hafta_3_2.cpp
#include <opencv2/opencv.hpp>
#include <iostream>
using namespace cv;
using namespace std;

int main() {
    Mat img = imread("/content/hafta_3.png", IMREAD_GRAYSCALE);

    if (img.empty()) {
        cout << "Error: Image not found!" << endl;
        return 1;
    }

    int histogram[256] = {0};

    for (int y = 0; y < img.rows; y++) {
        for (int x = 0; x < img.cols; x++) {
            int pixel = img.at<uchar>(y, x);
            histogram[pixel]++;
        }
    }

    // 3. Calculate the CDF manually
    int cdf[256] = {0};
    cdf[0] = histogram[0];

    for (int i = 1; i < 256; i++) {
        cdf[i] = cdf[i - 1] + histogram[i];
    }

    // Find the first non-zero CDF value
    int cdfMin = 0;

    for (int i = 0; i < 256; i++) {
        if (cdf[i] > 0) {
            cdfMin = cdf[i];
            break;
        }
    }

    Mat result(img.rows, img.cols, CV_8UC1);
    int totalPixels = img.rows * img.cols;

    for (int y = 0; y < img.rows; y++) {
        for (int x = 0; x < img.cols; x++) {
            int oldPixel = img.at<uchar>(y, x);

            int newPixel = 0;

            if (totalPixels > cdfMin) {
                newPixel = (int)(
                    (double)(cdf[oldPixel] - cdfMin) * 255
                    / (totalPixels - cdfMin)
                );
            }

            result.at<uchar>(y, x) = (uchar)newPixel;
        }
    }

    imwrite("/content/hafta_3_2_equalized.png", result);

    cout << "Image size: " << img.cols << " x " << img.rows << endl;
    cout << "Total pixels: " << totalPixels << endl;
    cout << "CDF minimum: " << cdfMin << endl;

    cout << "\nParlaklık\tHistogram\tCDF" << endl;

    for (int i = 0; i < 256; i++) {
        cout << i << "\t\t"
             << histogram[i] << "\t\t"
             << cdf[i] << endl;
    }

    return 0;
}


Overwriting hafta_3_2.cpp


In [50]:
!g++ hafta_3_2.cpp -o hafta_3_2 `pkg-config --cflags --libs opencv4`
!./hafta_3_2

Image size: 3000 x 2000
Total pixels: 6000000
CDF minimum: 1

Parlaklık	Histogram	CDF
0		0		0
1		0		0
2		0		0
3		0		0
4		0		0
5		0		0
6		0		0
7		0		0
8		0		0
9		0		0
10		0		0
11		0		0
12		0		0
13		0		0
14		0		0
15		0		0
16		0		0
17		0		0
18		0		0
19		0		0
20		0		0
21		0		0
22		0		0
23		0		0
24		0		0
25		0		0
26		0		0
27		0		0
28		0		0
29		0		0
30		0		0
31		0		0
32		0		0
33		0		0
34		0		0
35		0		0
36		0		0
37		0		0
38		0		0
39		0		0
40		0		0
41		0		0
42		0		0
43		0		0
44		0		0
45		0		0
46		0		0
47		0		0
48		0		0
49		0		0
50		0		0
51		0		0
52		0		0
53		0		0
54		0		0
55		0		0
56		0		0
57		0		0
58		0		0
59		0		0
60		0		0
61		0		0
62		0		0
63		0		0
64		0		0
65		0		0
66		0		0
67		0		0
68		0		0
69		0		0
70		0		0
71		0		0
72		0		0
73		0		0
74		0		0
75		0		0
76		0		0
77		0		0
78		0		0
79		0		0
80		0		0
81		0		0
82		0		0
83		0		0
84		0		0
85		0		0
86		0		0
87		0		0
88		0		0
89		0		0
90		0		0
91		0		0
92		0		0
93		0		0
94		0		0
95		0		0
96		0		0
97		0		0
98		0		0
99		0		0
100		1		1
101		4		5
102	

**Hafta_3_3 : Düşük kontrastlı ve tek kanallı bir görüntüyü OpenCV kullanarak açınız. OpenCV içindeki CLAHE (Contrast Limited Adaptive Histogram Equalization) hazır fonksiyonu ile resmi düzeltin. Daha sonra bu fonksiyonu hazır bir fonksiyon kullanmadan gerçekleyin ve test edin.**

In [51]:

%%writefile hafta_3_3_a.cpp
#include <opencv2/opencv.hpp>
#include <iostream>
using namespace cv;
using namespace std;

int main() {
    Mat img = imread("/content/hafta_3.png", IMREAD_GRAYSCALE);

    if (img.empty()) {
        cout << "Error: Image not found!" << endl;
        return 1;
    }

    Ptr<CLAHE> clahe = createCLAHE();

    Mat result;
    clahe->apply(img, result);

    imwrite("/content/hafta_3_3_a_clahe_result.png", result);

    return 0;
}


Writing hafta_3_3_a.cpp


In [52]:
!g++ hafta_3_3_a.cpp -o hafta_3_3_a `pkg-config --cflags --libs opencv4`
!./hafta_3_3_a

In [55]:

%%writefile hafta_3_3_b.cpp
#include <opencv2/opencv.hpp>
#include <iostream>
#include <algorithm>
using namespace cv;
using namespace std;

int main() {
    Mat img = imread("/content/hafta_3.png", IMREAD_GRAYSCALE);

    if (img.empty()) {
        cout << "Image not found!" << endl;
        return 1;
    }

    int tilesX = 8, tilesY = 8;
    int clipLimit = 40;

    int mapping[8][8][256] = {};

    for (int ty = 0; ty < tilesY; ty++) {
        for (int tx = 0; tx < tilesX; tx++) {
            int xStart = tx * img.cols / tilesX;
            int xEnd = (tx + 1) * img.cols / tilesX;
            int yStart = ty * img.rows / tilesY;
            int yEnd = (ty + 1) * img.rows / tilesY;

            int histogram[256] = {};

            for (int y = yStart; y < yEnd; y++) {
                for (int x = xStart; x < xEnd; x++) {
                    histogram[img.at<uchar>(y, x)]++;
                }
            }

            int excess = 0;
            for (int i = 0; i < 256; i++) {
              int tileArea = (xEnd - xStart) * (yEnd - yStart);
              int actualClipLimit = max(1, clipLimit * tileArea / 256);

              int excess = 0;

              for (int i = 0; i < 256; i++) {
                  if (histogram[i] > actualClipLimit) {
                      excess += histogram[i] - actualClipLimit;
                      histogram[i] = actualClipLimit;
                  }
              }
            }

            int each = excess / 256;
            int remainder = excess % 256;

            for (int i = 0; i < 256; i++)
                histogram[i] += each;

            for (int i = 0; i < remainder; i++)
                histogram[i]++;

            int cdf[256];
            cdf[0] = histogram[0];

            for (int i = 1; i < 256; i++)
                cdf[i] = cdf[i - 1] + histogram[i];

            int cdfMin = 0;
            for (int i = 0; i < 256; i++) {
                if (cdf[i] > 0) {
                    cdfMin = cdf[i];
                    break;
                }
            }

            int total = (xEnd - xStart) * (yEnd - yStart);

            for (int i = 0; i < 256; i++) {
                if (total > cdfMin) {
                    mapping[ty][tx][i] = (int)(
                        (double)(cdf[i] - cdfMin) * 255
                        / (total - cdfMin)
                    );
                } else {
                    mapping[ty][tx][i] = i;
                }
            }
        }
    }

    Mat result(img.rows, img.cols, CV_8UC1);

    for (int y = 0; y < img.rows; y++) {
        for (int x = 0; x < img.cols; x++) {
            double gx = ((x + 0.5) * tilesX / img.cols) - 0.5;
            double gy = ((y + 0.5) * tilesY / img.rows) - 0.5;

            int x0 = (int)floor(gx);
            int y0 = (int)floor(gy);

            double dx = gx - x0;
            double dy = gy - y0;

            int x1 = x0 + 1;
            int y1 = y0 + 1;

            x0 = max(0, min(x0, tilesX - 1));
            x1 = max(0, min(x1, tilesX - 1));
            y0 = max(0, min(y0, tilesY - 1));
            y1 = max(0, min(y1, tilesY - 1));

            dx = max(0.0, min(dx, 1.0));
            dy = max(0.0, min(dy, 1.0));

            int pixel = img.at<uchar>(y, x);

            double top =
                mapping[y0][x0][pixel] * (1.0 - dx) +
                mapping[y0][x1][pixel] * dx;

            double bottom =
                mapping[y1][x0][pixel] * (1.0 - dx) +
                mapping[y1][x1][pixel] * dx;

            double value = top * (1.0 - dy) + bottom * dy;

            result.at<uchar>(y, x) = (uchar)value;
        }
    }

    imwrite("/content/hafta_3_3_b_clahe_manual.png", result);
    cout << "Manual CLAHE result saved." << endl;

    return 0;
}


Overwriting hafta_3_3_b.cpp


In [56]:
!g++ hafta_3_3_b.cpp -o hafta_3_3_b `pkg-config --cflags --libs opencv4`
!./hafta_3_3_b

Manual CLAHE result saved.


**Hafta_3_4 : Bir gürültülü resim üzerinde 3x3 boyutunda ortalama almayı sağlayan bir filtreyi konvolüsyon ile gezdirin ve resimdeki gürültüyü bastırın.**

In [60]:
%%writefile hafta_3_4.cpp
#include <opencv2/opencv.hpp>
#include <iostream>
using namespace cv;
using namespace std;

int main() {
    Mat img = imread("/content/hafta_3_4_image.png", IMREAD_GRAYSCALE);

    if (img.empty()) {
        cout << "Image not found!" << endl;
        return 1;
    }

    Mat result = img.clone();

    for (int y = 1; y < img.rows - 1; y++) {
        for (int x = 1; x < img.cols - 1; x++) {
            int sum = 0;

            for (int ky = -1; ky <= 1; ky++) {
                for (int kx = -1; kx <= 1; kx++) {
                    sum += img.at<uchar>(y + ky, x + kx);
                }
            }

            result.at<uchar>(y, x) = sum / 9;
        }
    }

    imwrite("/content/hafta_3_4_filtered.png", result);

    return 0;
}


Overwriting hafta_3_4.cpp


In [61]:
!g++ hafta_3_4.cpp -o hafta_3_4 `pkg-config --cflags --libs opencv4`
!./hafta_3_4

**Hafte_3_5 : Salt-and-Pepper gürültülü bir resim bulunuz. Bu resim içindeki gürültüyü bastırmak için öncelikle OpenCV'de yer alan medyan filtre fonksiyonunu kullanınız. Daha sonra medyan filtreyi hazır fonksiyon kullanmadan kendiniz yazınız. Filtreniz 5x5 boyutunda olmalıdır.**


In [64]:

%%writefile hafta_3_5_a.cpp
#include <opencv2/opencv.hpp>
#include <iostream>
using namespace cv;
using namespace std;

int main() {
    Mat img = imread("/content/hafta_3_5_salt_pepper.png", IMREAD_GRAYSCALE);

    if (img.empty()) {
        cout << "Image not found!" << endl;
        return 1;
    }

    Mat result;

    medianBlur(img, result, 3);

    imwrite("/content/hafta_3_5_a_salt_pepper_filtered.png", result);

    return 0;
}


Overwriting hafta_3_5_a.cpp


In [65]:
!g++ hafta_3_5_a.cpp -o hafta_3_5_a `pkg-config --cflags --libs opencv4`
!./hafta_3_5_a

libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


hazır fonksiyon kullanmadan

In [67]:

%%writefile hafta_3_5_manual.cpp
#include <opencv2/opencv.hpp>
#include <iostream>
#include <algorithm>
using namespace cv;
using namespace std;

int main() {
    Mat img = imread("/content/hafta_3_5_salt_pepper.png", IMREAD_GRAYSCALE);

    if (img.empty()) {
        cout << "Image not found!" << endl;
        return 1;
    }

    Mat result = img.clone();

    for (int y = 1; y < img.rows - 1; y++) {
        for (int x = 1; x < img.cols - 1; x++) {
            uchar values[9];
            int k = 0;

            for (int ky = -1; ky <= 1; ky++) {
                for (int kx = -1; kx <= 1; kx++) {
                    values[k++] = img.at<uchar>(y + ky, x + kx);
                }
            }

            sort(values, values + 9);

            result.at<uchar>(y, x) = values[4];
        }
    }

    imwrite("/content/hafta_3_5_a_salt_pepper_manual_filtered.png", result);


    return 0;
}


Overwriting hafta_3_5_manual.cpp


In [68]:
!g++ hafta_3_5_manual.cpp -o hafta_3_5_manual `pkg-config --cflags --libs opencv4`
!./hafta_3_5_manual

libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


In [69]:
import zipfile
import glob
import os

files = glob.glob("/content/*.cpp") + glob.glob("/content/*.png")

with zipfile.ZipFile("/content/hafta_odevleri.zip", "w") as zipf:
    for file in files:
        zipf.write(file, os.path.basename(file))

print("ZIP file created!")
print("Files included:", len(files))

ZIP file created!
Files included: 33
